## Multi-MCP Agent: Microsoft Learn + Orders & Complaints

This notebook demonstrates an agent with **two MCP tools**:
1. **Microsoft Learn MCP** — for Azure documentation queries
2. **Orders & Complaints MCP** — local server (`http://localhost:8700/mcp`) for customer order and complaint management

**Prerequisite**: The Orders & Complaints MCP server must be running on port 8700.
```bash
cd use-cases-day4/mcp && python main.py
```

In [1]:
import os
from dotenv import load_dotenv
from agent_framework.openai import OpenAIChatClient
from azure.identity.aio import AzureCliCredential
from agent_framework import MCPStreamableHTTPTool

In [2]:
load_dotenv(override=True)

azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
model = os.getenv("AZURE_OPENAI_RESPONSES_DEPLOYMENT_NAME")

print("Azure OpenAI Endpoint: ", azure_endpoint)
print("Model: ", model)

Azure OpenAI Endpoint:  https://ramkumar-foundry-v19.services.ai.azure.com
Model:  gpt-4o


In [3]:
ms_learn_mcp_tool = MCPStreamableHTTPTool(
    name="Microsoft Learn MCP Tool",
    url="https://learn.microsoft.com/api/mcp"
)

orders_complaints_mcp_tool = MCPStreamableHTTPTool(
    name="Orders and Complaints MCP Tool",
    url="http://localhost:8700/mcp"
)

In [4]:
credential = AzureCliCredential()
client = OpenAIChatClient(
    model=model,
    azure_endpoint=azure_endpoint,
    credential=credential,
)

agent = client.as_agent(
    name="CustomerServiceAgent",
    instructions=(
        "You are a Customer Service Agent with access to two systems:\n"
        "1. Microsoft Learn Documentation - for looking up Azure and Microsoft product documentation.\n"
        "2. Orders & Complaints Management - for querying customer orders, registering complaints, "
        "and resolving complaints.\n\n"
        "When registering a complaint, use context from previous conversation turns to select the "
        "appropriate order_id and compose a relevant complaint description. "
        "Always confirm the action taken and include relevant IDs in your response."
    ),
    tools=[ms_learn_mcp_tool, orders_complaints_mcp_tool]
)

### Scenario 1: Get Order Details for Priya Sharma

In [5]:
session = agent.create_session()

query = "Get all order details for customer Priya Sharma"
response = await agent.run(query, session=session)
print("\nAssistant:\n", response)


Assistant:
 I found one order for Priya Sharma. Here are the details:

- **Order ID**: ORD10002
- **Order Date**: January 13, 2025
- **Product**: Xbox Wireless Controller
- **SKU**: MSXBXCTL
- **Quantity**: 1
- **Unit Price**: ₹5,490
- **Billing Address**: 45, Connaught Place, New Delhi, 110001
- **Order Status**: Processing
- **Remarks**: Order for Xbox Wireless Controller

Let me know if you need assistance regarding this order!


### Scenario 2a: Query Microsoft Learn Docs

In [6]:
query = "Using Microsoft Learn documentation, explain how to create an Azure Storage Account using Azure CLI"
response = await agent.run(query, session=session)
print("\nAssistant:\n", response)


Assistant:
 To create an Azure Storage Account using Azure CLI, you can follow these steps:

### Prerequisites:
1. Ensure you have the Azure CLI installed locally or access the Azure Cloud Shell in the Azure portal.
2. Log in to your Azure account:
   ```azurecli
   az login
   ```

### Steps to Create a Storage Account:

1. **Create a Resource Group**:
   Run the `az group create` command to create a new resource group where the storage account will reside. Replace `<resource-group-name>` and `<location>` with your desired values:
   ```azurecli
   az group create \
       --name <resource-group-name> \
       --location <location>
   ```

   You can list available Azure locations using:
   ```azurecli
   az account list-locations \
       --query ".[].{Region:name}" \
       --out table
   ```

2. **Create the Storage Account**:
   Use the `az storage account create` command to create a Standard general-purpose v2 storage account. Replace `<account-name>` with a unique storage accou

### Scenario 2b: Register a Complaint Based on MS Learn Response

In [7]:
query = (
    "Based on the Azure Storage Account documentation above, register a complaint "
    "for one of Priya Sharma's orders. The complaint should describe that the customer "
    "faced issues following the Azure CLI steps to create a storage account. "
    "Use High priority."
)
response = await agent.run(query, session=session)
print("\nAssistant:\n", response)


Assistant:
 The complaint has been successfully registered for Priya Sharma's order. Here are the details:

- **Complaint ID**: COMP10113
- **Order ID**: ORD10002
- **Description**: The customer faced issues following the Azure CLI steps to create an Azure Storage Account and encountered errors during the process. Further guidance or troubleshooting assistance is required.
- **Priority**: High
- **Status**: Open

We will address this issue promptly. Let me know if there’s anything else you’d like to add or inquire about!


### Scenario 3: Get All Complaints for Priya Sharma

In [8]:
query = "Get all complaints registered by Priya Sharma"
response = await agent.run(query, session=session)
print("\nAssistant:\n", response)


Assistant:
 Here are all the complaints registered by Priya Sharma:

1. **Complaint ID**: COMP10113
   - **Order ID**: ORD10002
   - **Date**: April 16, 2026
   - **Description**: The customer faced issues following the Azure CLI steps to create an Azure Storage Account and encountered errors.
   - **Priority**: High
   - **Status**: Open
   - **Assigned To**: Unassigned

2. **Complaint ID**: COMP10006
   - **Order ID**: ORD10002
   - **Date**: January 19, 2025
   - **Description**: Warranty card and invoice missing from the shipment.
   - **Priority**: Low
   - **Status**: Open
   - **Assigned To**: Raghav Menon

3. **Complaint ID**: COMP10010
   - **Order ID**: ORD10002
   - **Date**: January 19, 2025
   - **Description**: Order was cancelled without notification or consent.
   - **Priority**: Critical
   - **Status**: Escalated
   - **Assigned To**: Arun Kapoor

4. **Complaint ID**: COMP10008
   - **Order ID**: ORD10002
   - **Date**: January 18, 2025
   - **Description**: Missing 